In [1]:
!pip install polars

import polars as pl
import os
import random
import sys
import cv2
from tqdm import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import pandas as pd
from PIL import Image
import os
from torchvision import transforms
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random


In [2]:
import os
os.chdir('/workspace')

In [3]:
BASE_PATH = 'vindr_mammogram'
IMG_DIR = os.path.join(BASE_PATH, 'mammo_processed_cropped') 
df=pd.read_csv(f'{IMG_DIR}/mammo_processed_cropped.csv')
df = df.drop_duplicates(subset=['study_id', 'image_id'])
df["processed_path"] = (
    df["processed_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False))

/tmp/ipykernel_2133/2858854542.py:3: DtypeWarning: Columns (0: finding_birads) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(f'{IMG_DIR}/mammo_processed_cropped.csv')


In [4]:
df.head()

,study_id,series_id,image_id,laterality,view_position,height,width,breast_birads,breast_density,finding_categories,...,adjusted_ymin,adjusted_xmax,adjusted_ymax,roi_width,roi_height,processed_path,roi_xmin,roi_ymin,roi_xmax,roi_ymax
0,48575a27b7c992427041a82fa750d3fa,26de4993fa6b8ae50a91c8baf49b92b0,4e3a578fe535ea4f5258d3f7f4419db8,R,CC,3518,2800,BI-RADS 4,DENSITY C,['Mass'],...,1280.640015,720.979980,1401.750000,963,1971,vindr_mammogram/mammo_processed_cropped/48575a...,518.139893,1280.640015,645.979980,1401.750000
1,48575a27b7c992427041a82fa750d3fa,26de4993fa6b8ae50a91c8baf49b92b0,dac39351b0f3a8c670b7f8dc88029364,R,MLO,3518,2800,BI-RADS 4,DENSITY C,['Mass'],...,1240.609985,771.800049,1354.040039,1049,2229,vindr_mammogram/mammo_processed_cropped/48575a...,635.679932,1240.609985,750.800049,1354.040039
2,75e8e48933289d70b407379a564f8594,853b70e7e6f39133497909d9ca4c756d,c83f780904f25eacb44e9030f32c66e1,R,CC,3518,2800,BI-RADS 3,DENSITY C,['Global Asymmetry'],...,638.510010,816.439941,1656.260010,834,1953,vindr_mammogram/mammo_processed_cropped/75e8e4...,313.179932,638.510010,738.439941,1656.260010
3,75e8e48933289d70b407379a564f8594,853b70e7e6f39133497909d9ca4c756d,893528bc38a0362928a89364f1b692fd,R,MLO,3518,2800,BI-RADS 3,DENSITY C,['Global Asymmetry'],...,1443.640015,906.760010,2193.810059,1061,2372,vindr_mammogram/mammo_processed_cropped/75e8e4...,215.270020,1443.640015,850.760010,2193.810059
4,c3487424fee1bdd4515b72dc3fd69813,77619c914263eae44e9099f1ce07192c,318264c881bf12f2c1efe5f93920cc37,R,CC,3518,2800,BI-RADS 4,DENSITY C,['Architectural Distortion'],...,1533.410034,728.699951,1713.159912,1140,2115,vindr_mammogram/mammo_processed_cropped/c34874...,512.300049,1505.410034,728.699951,1685.159912


In [5]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 0
        else:
            return 1 if max_birads == 3 else 4
    
    if has_structural:
        return 4
    
    if has_mass and has_calc:
        return 3
    
    if has_mass:
        return 2
    
    if has_calc:
        return 1
    
    if has_lymph:
        return 3
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 4
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 4

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['finding_categories'], 
        row['finding_categories'],
        row['breast_birads'],
        row['breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    18070
1      403
2      928
3       89
4      192
Name: count, dtype: int64

In [6]:

selected_columns = ['processed_path','study_id', 'view_position', 'laterality', 'breast_birads','split']
df = df[selected_columns].copy()
def birads_to_binary(birads):
    return 0 if birads in ['BI-RADS 1'] else 1 
df['label'] = df['breast_birads'].apply(birads_to_binary)
df.head()

,processed_path,study_id,view_position,laterality,breast_birads,split,label
0,vindr_mammogram/mammo_processed_cropped/48575a...,48575a27b7c992427041a82fa750d3fa,CC,R,BI-RADS 4,training,1
1,vindr_mammogram/mammo_processed_cropped/48575a...,48575a27b7c992427041a82fa750d3fa,MLO,R,BI-RADS 4,training,1
4,vindr_mammogram/mammo_processed_cropped/c34874...,c3487424fee1bdd4515b72dc3fd69813,CC,R,BI-RADS 4,training,1
5,vindr_mammogram/mammo_processed_cropped/c34874...,c3487424fee1bdd4515b72dc3fd69813,MLO,R,BI-RADS 4,training,1
6,vindr_mammogram/mammo_processed_cropped/568385...,5683854eafabc34f6d854000d2ac6c2d,CC,L,BI-RADS 3,test,1


In [7]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['birads'] = df['breast_birads'].apply(birads_to_label)

In [8]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [9]:
df.head()

,processed_path,study_id,view_position,laterality,breast_birads,split,label,birads
0,vindr_mammogram/mammo_processed_cropped/48575a...,48575a27b7c992427041a82fa750d3fa,CC,R,BI-RADS 4,training,1,3
1,vindr_mammogram/mammo_processed_cropped/48575a...,48575a27b7c992427041a82fa750d3fa,MLO,R,BI-RADS 4,training,1,3
4,vindr_mammogram/mammo_processed_cropped/c34874...,c3487424fee1bdd4515b72dc3fd69813,CC,R,BI-RADS 4,training,1,3
5,vindr_mammogram/mammo_processed_cropped/c34874...,c3487424fee1bdd4515b72dc3fd69813,MLO,R,BI-RADS 4,training,1,3
6,vindr_mammogram/mammo_processed_cropped/568385...,5683854eafabc34f6d854000d2ac6c2d,CC,L,BI-RADS 3,test,1,2


In [10]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'breast_birads': 'first',   # BI-RADS at study level
        'breast_birads': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['breast_birads'].astype(str) + '_' +
    study_meta['breast_birads'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [11]:
inbreast_df = pd.read_csv("inbreast_data/cropped_inbreast_metadata.csv")
inbreast_df["processed_path"] = (
    inbreast_df["processed_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False))
inbreast_df.head()

,File Name,Laterality,View,Bi-Rads,roi_width,roi_height,processed_path
0,22678622,R,CC,1,924,2043,inbreast_data/INbreast_processed/22678622_61b1...
1,22678646,L,CC,3,1016,2142,inbreast_data/INbreast_processed/22678646_61b1...
2,22678670,R,MLO,1,1008,2707,inbreast_data/INbreast_processed/22678670_61b1...
3,22678694,L,MLO,3,1021,2765,inbreast_data/INbreast_processed/22678694_61b1...
4,22614074,R,CC,5,2593,3702,inbreast_data/INbreast_processed/22614074_6bd2...


In [12]:
def extract_patient_id(processed_path):
    """Extract patient ID from processed path filename"""
    if pd.isna(processed_path):
        return None
    filename = os.path.basename(processed_path)
    parts = filename.split('_')
    return parts[1]  

In [13]:
selected_columns = ['processed_path','study_id', 'view_position', 'laterality', 'breast_birads','label']
inbreast_df['birads'] = inbreast_df['Bi-Rads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})

def birads_to_binary(birads):
    return 0 if birads in ['1'] else 1 
inbreast_df['label'] = inbreast_df['birads'].apply(birads_to_binary)
inbreast_df['birads'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['study_id'] = inbreast_df['processed_path'].apply(extract_patient_id)
inbreast_df = inbreast_df.rename(columns={
    'Laterality': 'laterality',
    'View': 'view_position',
    'birads': 'breast_birads'  
})

inbreast_df = inbreast_df[selected_columns].copy()

inbreast_df['birads']=inbreast_df['breast_birads'].copy()
inbreast_df.head()


,processed_path,study_id,view_position,laterality,breast_birads,label,birads
0,inbreast_data/INbreast_processed/22678622_61b1...,61b13c59bcba149e,CC,R,0,0,0
1,inbreast_data/INbreast_processed/22678646_61b1...,61b13c59bcba149e,CC,L,2,1,2
2,inbreast_data/INbreast_processed/22678670_61b1...,61b13c59bcba149e,MLO,R,0,0,0
3,inbreast_data/INbreast_processed/22678694_61b1...,61b13c59bcba149e,MLO,L,2,1,2
4,inbreast_data/INbreast_processed/22614074_6bd2...,6bd24a0a42c19ce1,CC,R,4,1,4


In [14]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(256, 256)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=8,
                translate=(0.05, 0.05),
                scale=(0.9, 1.05),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def add_speckle_noise(image, std=0.03):
    """Speckle noise - multiplicative noise common in mammography"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, img_array.shape)
        noisy_img = img_array + img_array * noise
        return Image.fromarray((np.clip(noisy_img, 0, 1) * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [15]:
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import models, transforms
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import label_binarize
import os
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# 1. TWO-VIEW DATASET CLASS
# ─────────────────────────────────────────────────────────────────────────────

class MammoTwoViewDataset(Dataset):
    """
    Dataset that pairs CC and MLO views for the same study_id and laterality.
    If either CC or MLO is abnormal, the final label = 1.
    """
    def __init__(self, df, transform=None, label_col='label'):
        self.transform = transform
        self.label_col = label_col
        
        # Pivot to create paired rows (CC + MLO)
        paired = df.pivot_table(
            index=['study_id', 'laterality'],
            columns='view_position',
            values=['processed_path', label_col],
            aggfunc='first'
        ).reset_index()
        
        # Flatten MultiIndex columns
        paired.columns = ['_'.join(col).strip('_') for col in paired.columns.values]
        
        # Drop rows missing any view
        paired = paired.dropna(subset=['processed_path_CC', 'processed_path_MLO']).reset_index(drop=True)
        
        # Convert labels to numeric
        paired[f'{label_col}_CC'] = pd.to_numeric(paired[f'{label_col}_CC'], errors='coerce')
        paired[f'{label_col}_MLO'] = pd.to_numeric(paired[f'{label_col}_MLO'], errors='coerce')
        
        # Label fusion logic: if any abnormal view (1 for binary, max for BI-RADS)
        paired[label_col] = paired[[f'{label_col}_CC', f'{label_col}_MLO']].max(axis=1).astype(int)
        
        # Store only relevant fields
        self.data = paired[['study_id', 'laterality', 'processed_path_CC', 'processed_path_MLO', label_col]]
    
    def __len__(self):
        return len(self.data)
    
    def _load_image(self, path, laterality, view):
        """Load mammogram image in RGB, flip if Left, crop for MLO."""
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"[Warning] Failed to load {path}: {e}")
            img = Image.fromarray(np.zeros((512, 512, 3), dtype=np.uint8))
        
        # Horizontal flip for left breast
        if laterality == 'L':
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        
        # Slight cropping for MLO view
        if view == 'MLO':
            width, height = img.size
            crop_top = int(height * 0.1)
            img = img.crop((0, crop_top, width, height))
        
        return img
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        cc_img = self._load_image(row['processed_path_CC'], row['laterality'], 'CC')
        mlo_img = self._load_image(row['processed_path_MLO'], row['laterality'], 'MLO')
        
        if self.transform:
            cc_img = self.transform(cc_img)
            mlo_img = self.transform(mlo_img)
        
        return {
            'cc': cc_img,
            'mlo': mlo_img,
            'label': torch.tensor(row[self.label_col], dtype=torch.long),
            'study_id': row['study_id']
        }


# ─────────────────────────────────────────────────────────────────────────────
# 2. TRANSFORMS
# ─────────────────────────────────────────────────────────────────────────────

def get_transforms(img_size=(512, 512)):
    """Get train and val transforms"""
    train_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    
    return train_transform, val_transform


# ─────────────────────────────────────────────────────────────────────────────
# 3. CREATE DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────

def create_all_loaders(train_df, val_df, test_df, inbreast_df, 
                       img_size=(512, 512), batch_size=8, label_col='label'):
    """Creates dataloaders for train/val/test/inbreast"""
    
    train_transform, val_transform = get_transforms(img_size)
    
    train_dataset = MammoTwoViewDataset(train_df, transform=train_transform, label_col=label_col)
    val_dataset = MammoTwoViewDataset(val_df, transform=val_transform, label_col=label_col)
    test_dataset = MammoTwoViewDataset(test_df, transform=val_transform, label_col=label_col)
    inbreast_dataset = MammoTwoViewDataset(inbreast_df, transform=val_transform, label_col=label_col)
    
    # Weighted sampler for training
    labels = train_dataset.data[label_col].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    
    β = 0.5
    class_weights = (1.0 / class_counts) ** β
    class_weights = class_weights / class_weights.sum() * len(unique_classes)
    
    sample_weights = class_weights[labels]
    print(f"\nClass counts: {dict(zip(unique_classes, class_counts))}")
    print(f"Smoothed class weights: {np.round(class_weights, 3)}")
    
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True
    )
    
    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, 
                              num_workers=6, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                            num_workers=6, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, 
                             num_workers=4, pin_memory=True)
    inbreast_loader = DataLoader(inbreast_dataset, batch_size=1, shuffle=False, 
                                 num_workers=4, pin_memory=True)
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
    print(f"InBreast samples: {len(inbreast_dataset)}")
    
    return train_loader, val_loader, test_loader, inbreast_loader


# ─────────────────────────────────────────────────────────────────────────────
# 4. GENERIC TWO-VIEW MODEL
# ─────────────────────────────────────────────────────────────────────────────

class MammoDualViewModel(nn.Module):
    """
    Dual-view mammography model.
    Compatible with ResNet, DenseNet, and EfficientNet backbones.
    Each view (CC, MLO) is processed by its own backbone.
    Features are adaptively fused via MLO weighting and classified jointly.
    """
    def __init__(self, backbone_class, backbone_weights=None, num_classes=5, feature_dim=512, dropout=0.3):
        super().__init__()

        # Instantiate both view-specific backbones
        self.cc_backbone = backbone_class(weights=backbone_weights)
        self.mlo_backbone = backbone_class(weights=backbone_weights)

        # ---- Identify feature extractor & feature dimension ----
        self.cc_extractor, num_features = self._get_feature_extractor(self.cc_backbone)
        self.mlo_extractor, _ = self._get_feature_extractor(self.mlo_backbone)

        # ---- Projection layers (map backbone features → latent space) ----
        self.cc_proj = nn.Sequential(
            nn.Linear(num_features, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )

        self.mlo_proj = nn.Sequential(
            nn.Linear(num_features, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )

        # ---- Adaptive weighting for MLO branch ----
        self.weight_net = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

        # ---- MLO branch processing ----
        self.mlo_processor = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True)
        )

        # ---- Final classification head ----
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim + 256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    # ==============================================================
    # Helper function to handle different backbone architectures
    # ==============================================================    
    def _get_feature_extractor(self, model):
        """
        Returns a model stripped of its classification head and its output feature size.
        Supports EfficientNet, ResNet, DenseNet, ConvNeXt.
        """
        # ---------- EfficientNet ----------
        if hasattr(model, "classifier") and hasattr(model.classifier, "in_features"):
            num_features = model.classifier.in_features
            model.classifier = nn.Identity()
            return model, num_features

        # ---------- EfficientNet (Sequential classifier) ----------
        if hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
            # Newer torchvision EfficientNets use classifier[1]
            num_features = model.classifier[-1].in_features if hasattr(model.classifier[-1], 'in_features') else model.classifier[0].in_features
            model.classifier = nn.Identity()
            return model, num_features

        # ---------- ConvNeXt ----------
        if hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
            if len(model.classifier) > 0 and hasattr(model.classifier[2], 'in_features'):
                num_features = model.classifier[2].in_features
                model.classifier = nn.Identity()
                return model, num_features

        # ---------- ResNet ----------
        if hasattr(model, "fc"):
            num_features = model.fc.in_features
            model.fc = nn.Identity()
            return model, num_features

        # ---------- DenseNet ----------
        if hasattr(model, "classifier") and hasattr(model, "features"):
            num_features = model.classifier.in_features
            model.classifier = nn.Identity()
            return model, num_features

        # ---------- Fallback ----------
        print("[Warning] Unknown architecture: defaulting to 1000 features.")
        return model, 1000

    # ==============================================================
    # Forward pass
    # ==============================================================    
    def forward(self, cc, mlo):
        # Extract features
        cc_feat = self.cc_extractor(cc)
        mlo_feat = self.mlo_extractor(mlo)

        # Global average pool if needed
        if cc_feat.dim() == 4:
            cc_feat = cc_feat.mean([-2, -1])
        if mlo_feat.dim() == 4:
            mlo_feat = mlo_feat.mean([-2, -1])

        # Project to latent space
        cc_feat = self.cc_proj(cc_feat)
        mlo_feat = self.mlo_proj(mlo_feat)

        # Adaptive weighting of MLO features
        mlo_weight = self.weight_net(mlo_feat)           # shape [B, 1]
        mlo_weighted = mlo_feat * mlo_weight             # scaled features

        # Process MLO branch
        mlo_processed = self.mlo_processor(mlo_weighted)

        # Concatenate CC + MLO representations
        combined = torch.cat([cc_feat, mlo_processed], dim=1)

        # Classification
        logits = self.classifier(combined)
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 5. INFERENCE FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def run_inference(model, dataloader, device, num_classes=2):
    """Run inference and return predictions"""
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []
    
    for batch in dataloader:
        cc_images = batch['cc'].to(device)
        mlo_images = batch['mlo'].to(device)
        labels = batch['label']
        
        logits = model(cc_images, mlo_images)
        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


# ─────────────────────────────────────────────────────────────────────────────
# 6. CALCULATE METRICS
# ─────────────────────────────────────────────────────────────────────────────

def get_metrics(labels, preds, probs, num_classes=2):
    """Calculate Precision, Recall, Macro-F1, AUC"""
    
    if num_classes == 2:
        # Binary classification
        precision = precision_score(labels, preds, pos_label=1, zero_division=0)
        recall = recall_score(labels, preds, pos_label=1, zero_division=0)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        accuracy = accuracy_score(labels, preds)
        try:
            auc = roc_auc_score(labels, probs[:, 1])
        except:
            auc = np.nan
    else:
        # Multi-class (BI-RADS 5-class)
        precision = precision_score(labels, preds, average='macro', zero_division=0)
        recall = recall_score(labels, preds, average='macro', zero_division=0)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        accuracy = accuracy_score(labels, preds)
        try:
            y_bin = label_binarize(labels, classes=list(range(num_classes)))
            auc = roc_auc_score(y_bin, probs, multi_class='ovr', average='macro')
        except:
            auc = np.nan
    
    return {
        'Accuracy': round(float(accuracy), 4),
        'Precision': round(float(precision), 4),
        'Recall': round(float(recall), 4),
        'Macro-F1': round(float(macro_f1), 4),
        'AUC': round(float(auc), 4) if not np.isnan(auc) else '—'
    }


# ─────────────────────────────────────────────────────────────────────────────
# 7. FIND CHECKPOINT PATHS
# ─────────────────────────────────────────────────────────────────────────────

def find_checkpoints(base_path='Thesis_updated_results', task='binary'):
    """Find checkpoint paths for two-view models"""
    
    if task == 'binary':
        folder_prefix = '2_view_binary'
    else:
        folder_prefix = '2_view_birads'
    
    common_paths = {
        'efficientnet_b3_256': [
            f'{base_path}/{folder_prefix}_256/efficientnet_b3/best_model.pth',
        ],
        'efficientnet_b3_512': [
            f'{base_path}/{folder_prefix}_512/efficientnet_b3/best_model.pth',
        ],
        'convnext_base_256': [
            f'{base_path}/{folder_prefix}_256/convnext_base/best_model.pth',
        ],
        'convnext_base_512': [
            f'{base_path}/{folder_prefix}_512/convnext_base/best_model.pth',
        ]
    }
    
    effnet_256_path = None
    effnet_512_path = None
    convnext_256_path = None
    convnext_512_path = None
    
    print(f"\nSearching for {task.upper()} checkpoints in {folder_prefix}...")
    
    for key, paths in common_paths.items():
        for path in paths:
            if os.path.exists(path):
                print(f"  ✓ Found: {key} -> {path}")
                if '256' in key and 'efficientnet' in key:
                    effnet_256_path = path
                elif '512' in key and 'efficientnet' in key:
                    effnet_512_path = path
                elif '256' in key and 'convnext' in key:
                    convnext_256_path = path
                elif '512' in key and 'convnext' in key:
                    convnext_512_path = path
            else:
                print(f"  ✗ Not found: {path}")
    
    models_config = [
        {
            'name': 'efficientnet_b3',
            'ckpt_256': effnet_256_path,
            'ckpt_512': effnet_512_path
        },
        {
            'name': 'convnext_base',
            'ckpt_256': convnext_256_path,
            'ckpt_512': convnext_512_path
        }
    ]
    
    return models_config


# ─────────────────────────────────────────────────────────────────────────────
# 8. MAIN EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_task(task_name, num_classes, models_config, loaders_dict, device='cuda'):
    """Evaluate all models for a task (binary or BI-RADS)"""
    
    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*100}")
    print(f"TASK: {task_name.upper()} ({num_classes}-class) - TWO-VIEW EVALUATION")
    print(f"{'='*100}\n")
    
    all_results = []
    
    for model_cfg in models_config:
        model_name = model_cfg['name']
        print(f"\n{'─'*90}")
        print(f"MODEL: {model_name.upper()}")
        print(f"{'─'*90}")
        
        # Get backbone function and weights based on model name
        if 'efficientnet' in model_name:
            backbone_fn = models.efficientnet_b3
            backbone_weights = models.EfficientNet_B3_Weights.DEFAULT
        else:  # convnext_base
            backbone_fn = models.convnext_base
            backbone_weights = models.ConvNeXt_Base_Weights.DEFAULT
        
        for resolution in ['256', '512']:
            ckpt_key = f'ckpt_{resolution}'
            ckpt_path = model_cfg.get(ckpt_key)
            
            if not ckpt_path or not os.path.exists(ckpt_path):
                print(f"  [{resolution}px] Checkpoint not found: {ckpt_path}")
                continue
            
            print(f"  [{resolution}px] Loading checkpoint...")
            model = MammoDualViewModel(
                backbone_class=backbone_fn, 
                backbone_weights=backbone_weights,
                num_classes=num_classes, 
                feature_dim=512, 
                dropout=0.3
            ).to(device)
            try:
                checkpoint = torch.load(ckpt_path, map_location=device)
                # Handle state dict keys that might have model. prefix
                if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                    state_dict = checkpoint['model_state_dict']
                else:
                    state_dict = checkpoint
                
                model.load_state_dict(state_dict, strict=False)
                print(f"  [{resolution}px] ✓ Loaded successfully")
            except Exception as e:
                print(f"  [{resolution}px] ✗ Error loading: {e}")
                continue
            
            # Evaluate on each split
            for split_name in ['test', 'inbreast']:
                loader_key = f'{split_name}_{resolution}'
                loader = loaders_dict.get(loader_key)
                
                if loader is None:
                    continue
                
                print(f"    Evaluating {split_name.upper()} ({resolution}px)...", end=' ', flush=True)
                
                try:
                    labels, preds, probs = run_inference(model, loader, device, num_classes)
                    metrics = get_metrics(labels, preds, probs, num_classes)
                    
                    result = {
                        'Model': model_name.upper(),
                        'Split': split_name.upper(),
                        'Resolution': f'{resolution}px',
                        'N': len(labels),
                        **metrics
                    }
                    all_results.append(result)
                    print(f"✓ (AUC={metrics['AUC']}, F1={metrics['Macro-F1']})")
                except Exception as e:
                    print(f"✗ Error: {e}")
                    continue
            
            del model
            torch.cuda.empty_cache()
    
    # Create summary DataFrame
    results_df = pd.DataFrame(all_results)
    
    print(f"\n{'='*100}")
    print(f"TWO-VIEW {task_name.upper()} RESULTS TABLE")
    print(f"{'='*100}\n")
    print(results_df.to_string(index=False))
    
    # Save to CSV
    os.makedirs('results', exist_ok=True)
    csv_path = f'results/twoview_{task_name}_results_summary.csv'
    results_df.to_csv(csv_path, index=False)
    print(f"\n✓ Saved to: {csv_path}\n")
    
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# 9. MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    
    print("\n" + "="*100)
    print("TWO-VIEW MAMMOGRAM EVALUATION - BINARY & BI-RADS")
    print("="*100)
    
    # ─ LOAD YOUR DATAFRAMES ──────────────────────────────────────────────────
    print("\nLoading data...")
    
    # Replace with your actual dataframes
    # train_df, val_df, test_df, inbreast_df should have columns:
    #   - study_id
    #   - laterality (L or R)
    #   - view_position (CC or MLO)
    #   - processed_path (image path)
    #   - label (0-1 for binary, 0-4 for BI-RADS)
    
    # ─ BINARY CLASSIFICATION ──────────────────────────────────────────────────
    print("\n" + "─"*100)
    print("BINARY CLASSIFICATION (256px and 512px)")
    print("─"*100)
    
    # Create loaders for binary 256px
    print("\nCreating dataloaders for BINARY (256px)...")
    bin_train_loader_256, bin_val_loader_256, bin_test_loader_256, bin_inbreast_loader_256 = \
        create_all_loaders(train_df, val_df, test_df, inbreast_df, 
                          img_size=(256, 256), batch_size=8, label_col='label')
    
    # Create loaders for binary 512px
    print("\nCreating dataloaders for BINARY (512px)...")
    bin_train_loader_512, bin_val_loader_512, bin_test_loader_512, bin_inbreast_loader_512 = \
        create_all_loaders(train_df, val_df, test_df, inbreast_df, 
                          img_size=(512, 512), batch_size=8, label_col='label')
    
    binary_loaders = {
        'test_256': bin_test_loader_256,
        'test_512': bin_test_loader_512,
        'inbreast_256': bin_inbreast_loader_256,
        'inbreast_512': bin_inbreast_loader_512,
    }
    
    # Find binary checkpoints (2_view_binary_256 and 2_view_binary_512)
    print("\nFinding BINARY checkpoints...")
    binary_models = find_checkpoints(base_path='Thesis_updated_results', task='binary')
    
    # Evaluate binary
    binary_results = evaluate_task('binary', 2, binary_models, binary_loaders, device='cuda')
    
    # ─ BI-RADS CLASSIFICATION ────────────────────────────────────────────────
    print("\n" + "─"*100)
    print("BI-RADS 5-CLASS CLASSIFICATION (256px and 512px)")
    print("─"*100)
    
    # Create loaders for BI-RADS 256px
    print("\nCreating dataloaders for BI-RADS (256px)...")
    birads_train_loader_256, birads_val_loader_256, birads_test_loader_256, birads_inbreast_loader_256 = \
        create_all_loaders(train_df, val_df, test_df, inbreast_df, 
                          img_size=(256, 256), batch_size=8, label_col='birads')
    
    # Create loaders for BI-RADS 512px
    print("\nCreating dataloaders for BI-RADS (512px)...")
    birads_train_loader_512, birads_val_loader_512, birads_test_loader_512, birads_inbreast_loader_512 = \
        create_all_loaders(train_df, val_df, test_df, inbreast_df, 
                          img_size=(512, 512), batch_size=8, label_col='birads')
    
    birads_loaders = {
        'test_256': birads_test_loader_256,
        'test_512': birads_test_loader_512,
        'inbreast_256': birads_inbreast_loader_256,
        'inbreast_512': birads_inbreast_loader_512,
    }
    
    # Find BI-RADS checkpoints (2_view_birads_256 and 2_view_birads_512)
    print("\nFinding BI-RADS checkpoints...")
    birads_models = find_checkpoints(base_path='Thesis_updated_results', task='birads')
    
    # Evaluate BI-RADS
    birads_results = evaluate_task('birads', 5, birads_models, birads_loaders, device='cuda')
    
    # ─ FINAL SUMMARY ──────────────────────────────────────────────────────────
    print("\n" + "="*100)
    print("EVALUATION COMPLETE!")
    print("="*100)
    print(f"\nResults saved:")
    print(f"  - results/twoview_binary_results_summary.csv")
    print(f"  - results/twoview_birads_results_summary.csv")
    print(f"{"="*100}\n")


TWO-VIEW MAMMOGRAM EVALUATION - BINARY & BI-RADS

Loading data...

────────────────────────────────────────────────────────────────────────────────────────────────────
BINARY CLASSIFICATION (256px and 512px)
────────────────────────────────────────────────────────────────────────────────────────────────────

Creating dataloaders for BINARY (256px)...

Class counts: {np.int64(0): np.int64(4814), np.int64(1): np.int64(2236)}
Smoothed class weights: [0.811 1.189]
Train samples: 7050
Val samples: 784
Test samples: 1959
InBreast samples: 187

Creating dataloaders for BINARY (512px)...

Class counts: {np.int64(0): np.int64(4814), np.int64(1): np.int64(2236)}
Smoothed class weights: [0.811 1.189]
Train samples: 7050
Val samples: 784
Test samples: 1959
InBreast samples: 187

Finding BINARY checkpoints...

Searching for BINARY checkpoints in 2_view_binary...
  ✓ Found: efficientnet_b3_256 -> Thesis_updated_results/2_view_binary_256/efficientnet_b3/best_model.pth
  ✓ Found: efficientnet_b3_512 

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# =============================================================================
# AttentionPool2d (your existing code)
# =============================================================================

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


# =============================================================================
# FindingAwareProtoModel with SHAPE DEBUGGING
# =============================================================================

class FindingAwareProtoModel(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone,
        emb_dim=128,
        dropout=0.4,
        debug=True,  # NEW: Enable debugging
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone = backbone
        self.debug = debug

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True
        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True
        elif "resnet" in self.backbone_name:
            self.num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.is_cnn = True
        elif "densenet" in self.backbone_name:
            self.num_features = backbone.classifier.in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True
        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head = nn.Identity()
            self.is_cnn = False
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        if self.debug:
            print(f"\n{'='*70}")
            print(f"Backbone: {backbone_name}")
            print(f"Backbone type: {'CNN' if self.is_cnn else 'Transformer'}")
            print(f"Feature dimension: {self.num_features}")
            print(f"{'='*70}\n")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        # Binary classification head
        self.binary_head = nn.Sequential(
            nn.Linear(self.num_features, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
        )

        # Finding projection head
        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        # BI-RADS projection head
        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for head in [self.binary_head, self.proj_head_finding, self.proj_head_birads]:
            for m in head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                    if hasattr(m, "weight") and m.weight is not None:
                        nn.init.ones_(m.weight)
                    if hasattr(m, "bias") and m.bias is not None:
                        nn.init.zeros_(m.bias)

    def _extract(self, x):
        """Extract features with detailed shape logging"""
        
        if self.debug:
            print(f"\n{'='*70}")
            print("FEATURE EXTRACTION PIPELINE")
            print(f"{'='*70}")
            print(f"Input shape:                    {x.shape}")  # [B, 3, H, W]

        # Step 1: Backbone forward
        f = self.backbone(x)
        if self.debug:
            print(f"After backbone:                {f.shape if not isinstance(f, tuple) else f[0].shape}")

        # Handle tuple output (some backbones return (features, pooled))
        if isinstance(f, tuple):
            f = f[0]
            if self.debug:
                print(f"Extracted from tuple:          {f.shape}")

        # Step 2: Pooling/flattening
        if self.is_cnn:
            if f.ndim == 4:  # [B, C, H, W]
                if self.debug:
                    print(f"4D tensor (CNN features):      {f.shape}")
                
                if self.pool is not None:
                    if self.debug:
                        print(f"Applying AttentionPool2d...")
                    f = self.pool(f)
                    if self.debug:
                        print(f"After AttentionPool2d:         {f.shape}")
                else:
                    if self.debug:
                        print(f"Applying global avg pooling...")
                    f = f.flatten(2).mean(-1)
                    if self.debug:
                        print(f"After global avg pooling:      {f.shape}")
            
            elif f.ndim == 3:  # [B, C, N] or [B, N, C]
                if self.debug:
                    print(f"3D tensor (intermediate):      {f.shape}")
                    print(f"Applying mean pooling...")
                f = f.mean(1)
                if self.debug:
                    print(f"After mean pooling:            {f.shape}")
        else:
            # Transformer path
            if f.ndim == 3:  # [B, N, C]
                if self.debug:
                    print(f"3D tensor (Transformer):       {f.shape}")
                    print(f"Applying mean pooling...")
                f = f.mean(1)
                if self.debug:
                    print(f"After mean pooling:            {f.shape}")
            
            elif f.ndim == 4:  # [B, C, H, W]
                if self.debug:
                    print(f"4D tensor (unexpected):        {f.shape}")
                    print(f"Applying spatial flattening...")
                f = f.flatten(2).mean(-1)
                if self.debug:
                    print(f"After flattening:              {f.shape}")

        if self.debug:
            print(f"\nFinal extracted features:      {f.shape}")
            print(f"Expected:                      [B, {self.num_features}]")
            assert f.shape[1] == self.num_features, f"Shape mismatch! Got {f.shape[1]}, expected {self.num_features}"
            print("✓ Shape validation passed!")
            print(f"{'='*70}\n")

        return f

    def forward(self, x, return_embeddings=False):
        feat = self._extract(x)
        
        # Binary head forward
        if self.debug:
            print(f"Binary head input:             {feat.shape}")
        binary_logit = self.binary_head(feat)
        if self.debug:
            print(f"Binary logits output:          {binary_logit.shape}")

        if return_embeddings:
            # Finding embeddings
            if self.debug:
                print(f"\nFinding projection input:      {feat.shape}")
            emb_f = F.normalize(self.proj_head_finding(feat), dim=1)
            if self.debug:
                print(f"Finding embedding output:      {emb_f.shape}")
            
            # BI-RADS embeddings
            if self.debug:
                print(f"\nBI-RADS projection input:      {feat.shape}")
            emb_b = F.normalize(self.proj_head_birads(feat), dim=1)
            if self.debug:
                print(f"BI-RADS embedding output:      {emb_b.shape}")
            
            if self.debug:
                print(f"{'='*70}\n")
            
            return binary_logit, emb_f, emb_b

        if self.debug:
            print(f"{'='*70}\n")

        return binary_logit


# =============================================================================
# TEST WITH DIFFERENT BACKBONES
# =============================================================================

def test_backbone(backbone_name, input_shape=(2, 3, 512, 512)):
    """Test a specific backbone"""
    
    print(f"\n\n{'#'*70}")
    print(f"TESTING: {backbone_name.upper()}")
    print(f"Input shape: {input_shape}")
    print(f"{'#'*70}")

    # Get backbone
    if "efficientnet" in backbone_name:
        backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
    elif "convnext" in backbone_name:
        backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.DEFAULT)
    elif "resnet" in backbone_name:
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    elif "densenet" in backbone_name:
        backbone = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    # Create model with debugging
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FindingAwareProtoModel(
        backbone_name=backbone_name,
        backbone=backbone,
        emb_dim=128,
        dropout=0.4,
        debug=True
    ).to(device)

    # Create dummy input
    x = torch.randn(input_shape).to(device)

    # Test without embeddings
    print("\n--- FORWARD PASS (Classification Only) ---")
    logits = model(x, return_embeddings=False)
    print(f"Output: {logits.shape}\n")

    # Test with embeddings
    print("\n--- FORWARD PASS (With Embeddings) ---")
    logits, emb_f, emb_b = model(x, return_embeddings=True)
    print(f"Logits: {logits.shape}")
    print(f"Finding emb: {emb_f.shape}")
    print(f"BI-RADS emb: {emb_b.shape}\n")


# =============================================================================
# RUN TESTS
# =============================================================================

if __name__ == "__main__":
    # Test different backbones
    backbones = [
        # "efficientnet_b3",
        # "convnext_base",
        "resnet50",
        "densenet121",
    ]

    for backbone_name in backbones:
        try:
            test_backbone(backbone_name)
        except Exception as e:
            print(f"Error testing {backbone_name}: {e}\n")

    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print("""
Shape progression:
    Input:            [B, 3, 512, 512]
         ↓ backbone
    Backbone output:  [B, num_channels, H, W] or [B, N, num_channels]
         ↓ pooling
    After pooling:    [B, num_features]
         ↓ heads
    Binary logits:    [B, 2]
    Embeddings:       [B, 128]

Key points:
    • AttentionPool2d reduces [B, C, H, W] → [B, C]
    • Projections map [B, num_features] → [B, 128]
    • Normalization applied after projection
    • All shapes validated during forward pass
    """)



######################################################################
TESTING: RESNET50
Input shape: (2, 3, 512, 512)
######################################################################
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 303MB/s]



Backbone: resnet50
Backbone type: CNN
Feature dimension: 2048


--- FORWARD PASS (Classification Only) ---

FEATURE EXTRACTION PIPELINE
Input shape:                    torch.Size([2, 3, 512, 512])
After backbone:                torch.Size([2, 2048])

Final extracted features:      torch.Size([2, 2048])
Expected:                      [B, 2048]
✓ Shape validation passed!

Binary head input:             torch.Size([2, 2048])
Binary logits output:          torch.Size([2, 2])

Output: torch.Size([2, 2])


--- FORWARD PASS (With Embeddings) ---

FEATURE EXTRACTION PIPELINE
Input shape:                    torch.Size([2, 3, 512, 512])
After backbone:                torch.Size([2, 2048])

Final extracted features:      torch.Size([2, 2048])
Expected:                      [B, 2048]
✓ Shape validation passed!

Binary head input:             torch.Size([2, 2048])
Binary logits output:          torch.Size([2, 2])

Finding projection input:      torch.Size([2, 2048])
Finding embedding output:      

100%|██████████| 30.8M/30.8M [00:00<00:00, 297MB/s]



Backbone: densenet121
Backbone type: CNN
Feature dimension: 1024


--- FORWARD PASS (Classification Only) ---

FEATURE EXTRACTION PIPELINE
Input shape:                    torch.Size([2, 3, 512, 512])
After backbone:                torch.Size([2, 1024])

Final extracted features:      torch.Size([2, 1024])
Expected:                      [B, 1024]
✓ Shape validation passed!

Binary head input:             torch.Size([2, 1024])
Binary logits output:          torch.Size([2, 2])

Output: torch.Size([2, 2])


--- FORWARD PASS (With Embeddings) ---

FEATURE EXTRACTION PIPELINE
Input shape:                    torch.Size([2, 3, 512, 512])
After backbone:                torch.Size([2, 1024])

Final extracted features:      torch.Size([2, 1024])
Expected:                      [B, 1024]
✓ Shape validation passed!

Binary head input:             torch.Size([2, 1024])
Binary logits output:          torch.Size([2, 2])

Finding projection input:      torch.Size([2, 1024])
Finding embedding output:   

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style("darkgrid")

# =============================================================================
# FINDING-LEVEL BINARY DATA (ONLY RECALL)
# =============================================================================

findings_binary_data = {
    'EffNet-B3': {
        'Mass': {'Recall': 0.6726},
        'Calcification': {'Recall': 0.8600},
        'Structural': {'Recall': 0.7333},
        'Lymph Node': {'Recall': 1.0000},
    },
    'ConvNeXt-B': {
        'Mass': {'Recall': 0.7168},
        'Calcification': {'Recall': 0.8400},
        'Structural': {'Recall': 0.6000},
        'Lymph Node': {'Recall': 1.0000},
    }
}

# =============================================================================
# BI-RADS DATA (UNCHANGED)
# =============================================================================

findings_birads_data = {
    'EffNet-B3': {
        'Mass': {'F1': 0.2741, 'Recall': 0.2153, 'AUC': 0.744},
        'Calcification': {'F1': 0.2106, 'Recall': 0.1744, 'AUC': 0.634},
        'Structural': {'F1': 0.2263, 'Recall': 0.1952, 'AUC': 0.5792},
        'Lymph Node': {'F1': 0.4000, 'Recall': 0.5000, 'AUC': 1.0},
    },
    'ConvNeXt-B': {
        'Mass': {'F1': 0.3012, 'Recall': 0.2402, 'AUC': 0.768},
        'Calcification': {'F1': 0.2345, 'Recall': 0.1944, 'AUC': 0.656},
        'Structural': {'F1': 0.2418, 'Recall': 0.2143, 'AUC': 0.6125},
        'Lymph Node': {'F1': 0.4200, 'Recall': 0.5200, 'AUC': 1.0},
    }
}

# =============================================================================
# PLOT: BINARY FINDING ANALYSIS (RECALL ONLY)
# =============================================================================

def plot_binary_findings_recall():
    fig, ax = plt.subplots(figsize=(12, 6))

    findings = list(findings_binary_data['EffNet-B3'].keys())
    x = np.arange(len(findings))
    width = 0.35

    recall_effnet = [findings_binary_data['EffNet-B3'][f]['Recall'] for f in findings]
    recall_convnext = [findings_binary_data['ConvNeXt-B'][f]['Recall'] for f in findings]

    bars1 = ax.bar(x - width/2, recall_effnet, width,
                   label='EffNet-B3', color='#1f77b4')
    bars2 = ax.bar(x + width/2, recall_convnext, width,
                   label='ConvNeXt-B', color='#ff7f0e')

    ax.set_xlabel("Finding Type", fontsize=12, fontweight='bold')
    ax.set_ylabel("Recall", fontsize=12, fontweight='bold')
    ax.set_title("Binary Classification - Recall by Finding Type", fontsize=13, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(findings)

    ax.set_ylim([0, 1.05])
    ax.legend()

    for i, v in enumerate(recall_effnet):
        ax.text(i - width/2, v + 0.02, f"{v:.3f}", ha='center', fontsize=10)

    for i, v in enumerate(recall_convnext):
        ax.text(i + width/2, v + 0.02, f"{v:.3f}", ha='center', fontsize=10)

    plt.tight_layout()
    os.makedirs("results", exist_ok=True)
    plt.savefig("/workspace/paper_results/binary_findings_recall.png", dpi=300)
    plt.close()


# =============================================================================
# PLOT: BI-RADS FINDINGS (UNCHANGED)
# =============================================================================

def plot_birads_findings_f1():
    fig, ax = plt.subplots(figsize=(12, 6))

    findings = list(findings_birads_data['EffNet-B3'].keys())
    x = np.arange(len(findings))
    width = 0.35

    f1_effnet = [findings_birads_data['EffNet-B3'][f]['F1'] for f in findings]
    f1_convnext = [findings_birads_data['ConvNeXt-B'][f]['F1'] for f in findings]

    ax.bar(x - width/2, f1_effnet, width, label='EffNet-B3', color='green')
    ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-B', color='red')

    ax.set_title("BI-RADS Classification - F1 by Finding Type")
    ax.set_ylabel("F1 Score")
    ax.set_xticks(x)
    ax.set_xticklabels(findings)
    ax.set_ylim([0, 0.5])
    ax.legend()

    plt.tight_layout()
    os.makedirs("results", exist_ok=True)
    plt.savefig("/workspace/paper_results/birads_findings_f1.png", dpi=300)
    plt.close()


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    plot_binary_findings_recall()
    plot_birads_findings_f1()
    print("Done: Binary uses Recall only, BI-RADS unchanged.")

Done: Binary uses Recall only, BI-RADS unchanged.


In [6]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# =============================================================================
# Data - Binary Classification
# =============================================================================

# Binary Classification by Device
device_binary_data = {
    'EffNet-B3': {
        'Mammomat': {'F1': 0.7797},
        'Planmed': {'F1': 0.7457},
        'GIOTTO': {'F1': 0.8961},
    },
    'ConvNeXt-B': {
        'Mammomat': {'F1': 0.7971},
        'Planmed': {'F1': 0.7906},
        'GIOTTO': {'F1': 0.8570},
    }
}

# Binary Classification by Breast Density
density_binary_data = {
    'EffNet-B3': {
        'Scattered (B)': {'F1': 0.7886},
        'Hetero. (C)': {'F1': 0.7822},
        'Dense (D)': {'F1': 0.7644},
    },
    'ConvNeXt-B': {
        'Scattered (B)': {'F1': 0.7986},
        'Hetero. (C)': {'F1': 0.8032},
        'Dense (D)': {'F1': 0.7837},
    }
}

# =============================================================================
# Data - BI-RADS 5-Class Classification
# =============================================================================

# BI-RADS Classification by Device
device_birads_data = {
    'EffNet-B3': {
        'Mammomat': {'F1': 0.5412},
        'Planmed': {'F1': 0.3633},
        'GIOTTO': {'F1': 0.5703},
    },
    'ConvNeXt-B': {
        'Mammomat': {'F1': 0.5673},
        'Planmed': {'F1': 0.4727},
        'GIOTTO': {'F1': 0.5296},
    }
}

# BI-RADS Classification by Breast Density
density_birads_data = {
    'EffNet-B3': {
        'Scattered (B)': {'F1': 0.6892},
        'Hetero. (C)': {'F1': 0.5000},
        'Dense (D)': {'F1': 0.4750},
    },
    'ConvNeXt-B': {
        'Scattered (B)': {'F1': 0.6759},
        'Hetero. (C)': {'F1': 0.5235},
        'Dense (D)': {'F1': 0.4778},
    }
}

# =============================================================================
# Binary Classification by Finding Type (Hard Cases Only)
# =============================================================================

findings_binary_data = {
    'EffNet-B3': {
        'Mass': {'F1': 0.4021, 'Recall': 0.6726},
        'Calcification': {'F1': 0.4624, 'Recall': 0.8600},
        'Structural': {'F1': 0.4231, 'Recall': 0.7333},
        'Lymph Node': {'F1': 1.0000, 'Recall': 1.0000},
    },
    'ConvNeXt-B': {
        'Mass': {'F1': 0.4175, 'Recall': 0.7168},
        'Calcification': {'F1': 0.4565, 'Recall': 0.8400},
        'Structural': {'F1': 0.3750, 'Recall': 0.6000},
        'Lymph Node': {'F1': 1.0000, 'Recall': 1.0000},
    }
}

# =============================================================================
# BI-RADS Classification by Finding Type (Hard Cases Only)
# From BI-RADS Finding Analysis Table
# =============================================================================

findings_birads_data = {
    'EffNet-B3': {
        'Mass': {
            'N': 113,
            'Correct': 30,
            'Incorrect': 83,
            'Accuracy': 0.2655,
            'F1': 0.2741,
            'Recall': 0.2153,
            'AUC': 0.744
        },
        'Calcification': {
            'N': 50,
            'Correct': 17,
            'Incorrect': 33,
            'Accuracy': 0.3400,
            'F1': 0.2106,
            'Recall': 0.1744,
            'AUC': 0.634
        },
        'Structural': {
            'N': 15,
            'Correct': 6,
            'Incorrect': 9,
            'Accuracy': 0.4000,
            'F1': 0.2263,
            'Recall': 0.1952,
            'AUC': 0.5792
        },
        'Lymph Node': {
            'N': 3,
            'Correct': 2,
            'Incorrect': 1,
            'Accuracy': 0.6667,
            'F1': 0.4000,
            'Recall': 0.5000,
            'AUC': 1.0
        },
    },
    'ConvNeXt-B': {
        'Mass': {
            'N': 113,
            'Correct': 35,
            'Incorrect': 78,
            'Accuracy': 0.3097,
            'F1': 0.3012,
            'Recall': 0.2402,
            'AUC': 0.768
        },
        'Calcification': {
            'N': 50,
            'Correct': 19,
            'Incorrect': 31,
            'Accuracy': 0.3800,
            'F1': 0.2345,
            'Recall': 0.1944,
            'AUC': 0.656
        },
        'Structural': {
            'N': 15,
            'Correct': 7,
            'Incorrect': 8,
            'Accuracy': 0.4667,
            'F1': 0.2418,
            'Recall': 0.2143,
            'AUC': 0.6125
        },
        'Lymph Node': {
            'N': 3,
            'Correct': 2,
            'Incorrect': 1,
            'Accuracy': 0.6667,
            'F1': 0.4200,
            'Recall': 0.5200,
            'AUC': 1.0
        },
    }
}


# =============================================================================
# Figure 1: Binary Classification - Device Performance
# =============================================================================

def plot_binary_device():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    devices = list(device_binary_data['EffNet-B3'].keys())
    x = np.arange(len(devices))
    width = 0.35
    
    # F1 Scores only
    f1_effnet = [device_binary_data['EffNet-B3'][d]['F1'] for d in devices]
    f1_convnext = [device_binary_data['ConvNeXt-B'][d]['F1'] for d in devices]
    
    bars1 = ax.bar(x - width/2, f1_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#1f77b4')
    bars2 = ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#ff7f0e')
    
    ax.set_xlabel('Imaging Device', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('Binary Classification - F1 Score by Imaging Device', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(devices, fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0.70, 0.92])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(f1_effnet):
        ax.text(i - width/2, v + 0.008, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(f1_convnext):
        ax.text(i + width/2, v + 0.008, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/binary_device_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/binary_device_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 2: Binary Classification - Breast Density Performance
# =============================================================================

def plot_binary_density():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    densities = list(density_binary_data['EffNet-B3'].keys())
    x = np.arange(len(densities))
    width = 0.35
    
    # F1 Scores only
    f1_effnet = [density_binary_data['EffNet-B3'][d]['F1'] for d in densities]
    f1_convnext = [density_binary_data['ConvNeXt-B'][d]['F1'] for d in densities]
    
    bars1 = ax.bar(x - width/2, f1_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#2ca02c')
    bars2 = ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#d62728')
    
    ax.set_xlabel('Breast Density', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('Binary Classification - F1 Score by Breast Density', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(densities, fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0.75, 0.82])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(f1_effnet):
        ax.text(i - width/2, v + 0.002, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(f1_convnext):
        ax.text(i + width/2, v + 0.002, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/binary_density_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/binary_density_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 3: Binary Classification - Finding Type Performance
# =============================================================================

def plot_binary_findings():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    findings = list(findings_binary_data['EffNet-B3'].keys())
    x = np.arange(len(findings))
    width = 0.35
    
    f1_effnet = [findings_binary_data['EffNet-B3'][f]['F1'] for f in findings]
    f1_convnext = [findings_binary_data['ConvNeXt-B'][f]['F1'] for f in findings]
    
    bars1 = ax.bar(x - width/2, f1_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#9467bd')
    bars2 = ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#8c564b')
    
    ax.set_xlabel('Hard Case - Radiological Finding Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('Binary Classification - F1 Score by Hard Case Finding Type', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(findings, rotation=15, ha='right', fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0, 0.5])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(f1_effnet):
        ax.text(i - width/2, v + 0.015, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(f1_convnext):
        ax.text(i + width/2, v + 0.015, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/binary_findings_f1_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/binary_findings_f1_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 4: BI-RADS Classification - Device Performance
# =============================================================================

def plot_birads_device():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    devices = list(device_birads_data['EffNet-B3'].keys())
    x = np.arange(len(devices))
    width = 0.35
    
    # F1 Scores only
    f1_effnet = [device_birads_data['EffNet-B3'][d]['F1'] for d in devices]
    f1_convnext = [device_birads_data['ConvNeXt-B'][d]['F1'] for d in devices]
    
    bars1 = ax.bar(x - width/2, f1_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#17becf')
    bars2 = ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#bcbd22')
    
    ax.set_xlabel('Imaging Device', fontsize=12, fontweight='bold')
    ax.set_ylabel('Macro-F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('BI-RADS 5-Class Classification - F1 Score by Imaging Device', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(devices, fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0.30, 0.65])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(f1_effnet):
        ax.text(i - width/2, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(f1_convnext):
        ax.text(i + width/2, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/birads_device_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/birads_device_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 5: BI-RADS Classification - Breast Density Performance
# =============================================================================

def plot_birads_density():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    densities = list(density_birads_data['EffNet-B3'].keys())
    x = np.arange(len(densities))
    width = 0.35
    
    # F1 Scores only
    f1_effnet = [density_birads_data['EffNet-B3'][d]['F1'] for d in densities]
    f1_convnext = [density_birads_data['ConvNeXt-B'][d]['F1'] for d in densities]
    
    bars1 = ax.bar(x - width/2, f1_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#e377c2')
    bars2 = ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#7f7f7f')
    
    ax.set_xlabel('Breast Density', fontsize=12, fontweight='bold')
    ax.set_ylabel('Macro-F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('BI-RADS 5-Class Classification - F1 Score by Breast Density', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(densities, fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0.45, 0.75])
    ax.grid(axis='y', alpha=0.3)
    
    # Add annotation for degradation pattern
    ax.annotate('Performance degrades\nwith density increase', xy=(2, 0.4750), xytext=(1.5, 0.55),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=10, color='red', fontweight='bold')
    
    # Add value labels
    for i, v in enumerate(f1_effnet):
        ax.text(i - width/2, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(f1_convnext):
        ax.text(i + width/2, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/birads_density_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/birads_density_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 6: BI-RADS Classification - Finding Type Accuracy
# =============================================================================

def plot_birads_findings_accuracy():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    findings = ['Mass', 'Calcification', 'Structural', 'Lymph Node']
    x = np.arange(len(findings))
    width = 0.35
    
    acc_effnet = [findings_birads_data['EffNet-B3'][f]['Accuracy'] for f in findings]
    acc_convnext = [findings_birads_data['ConvNeXt-B'][f]['Accuracy'] for f in findings]
    
    bars1 = ax.bar(x - width/2, acc_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#1f77b4')
    bars2 = ax.bar(x + width/2, acc_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#ff7f0e')
    
    ax.set_xlabel('Hard Case - Radiological Finding Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('Accuracy (Correct/Total)', fontsize=12, fontweight='bold')
    ax.set_title('BI-RADS 5-Class Classification - Accuracy by Hard Case Finding Type', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(findings, rotation=15, ha='right', fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0, 0.8])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels with percentage
    for i, v in enumerate(acc_effnet):
        ax.text(i - width/2, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(acc_convnext):
        ax.text(i + width/2, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Add sample counts on x-axis
    sample_counts = ['N=113', 'N=50', 'N=15', 'N=3']
    for i, count in enumerate(sample_counts):
        ax.text(i, -0.08, count, ha='center', fontsize=9, style='italic', transform=ax.get_xaxis_transform())
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/birads_findings_accuracy_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/birads_findings_accuracy_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 7: BI-RADS Classification - Finding Type F1 Score
# =============================================================================

def plot_birads_findings_f1():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    findings = ['Mass', 'Calcification', 'Structural', 'Lymph Node']
    x = np.arange(len(findings))
    width = 0.35
    
    f1_effnet = [findings_birads_data['EffNet-B3'][f]['F1'] for f in findings]
    f1_convnext = [findings_birads_data['ConvNeXt-B'][f]['F1'] for f in findings]
    
    bars1 = ax.bar(x - width/2, f1_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#2ca02c')
    bars2 = ax.bar(x + width/2, f1_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#d62728')
    
    ax.set_xlabel('Hard Case - Radiological Finding Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score (Macro-F1)', fontsize=12, fontweight='bold')
    ax.set_title('BI-RADS 5-Class Classification - F1 Score by Hard Case Finding Type', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(findings, rotation=15, ha='right', fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0, 0.5])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(f1_effnet):
        ax.text(i - width/2, v + 0.015, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(f1_convnext):
        ax.text(i + width/2, v + 0.015, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Add sample counts on x-axis
    sample_counts = ['N=113', 'N=50', 'N=15', 'N=3']
    for i, count in enumerate(sample_counts):
        ax.text(i, -0.08, count, ha='center', fontsize=9, style='italic', transform=ax.get_xaxis_transform())
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/birads_findings_f1_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/birads_findings_f1_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Figure 8: BI-RADS Finding Type - AUC Score
# =============================================================================

def plot_birads_findings_auc():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    findings = ['Mass', 'Calcification', 'Structural', 'Lymph Node']
    x = np.arange(len(findings))
    width = 0.35
    
    auc_effnet = [findings_birads_data['EffNet-B3'][f]['AUC'] for f in findings]
    auc_convnext = [findings_birads_data['ConvNeXt-B'][f]['AUC'] for f in findings]
    
    bars1 = ax.bar(x - width/2, auc_effnet, width, label='EfficientNet-B3', alpha=0.85, color='#9467bd')
    bars2 = ax.bar(x + width/2, auc_convnext, width, label='ConvNeXt-Base', alpha=0.85, color='#8c564b')
    
    ax.set_xlabel('Hard Case - Radiological Finding Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('AUC Score', fontsize=12, fontweight='bold')
    ax.set_title('BI-RADS 5-Class Classification - AUC by Hard Case Finding Type', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(findings, rotation=15, ha='right', fontsize=11)
    ax.legend(fontsize=11)
    ax.set_ylim([0.5, 1.05])
    ax.grid(axis='y', alpha=0.3)
    
    # Add reference line for perfect AUC
    ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Perfect AUC (1.0)')
    ax.legend(fontsize=11)
    
    # Add value labels
    for i, v in enumerate(auc_effnet):
        ax.text(i - width/2, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for i, v in enumerate(auc_convnext):
        ax.text(i + width/2, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Add sample counts on x-axis
    sample_counts = ['N=113', 'N=50', 'N=15', 'N=3']
    for i, count in enumerate(sample_counts):
        ax.text(i, 0.48, count, ha='center', fontsize=9, style='italic')
    
    plt.tight_layout()
    os.makedirs('/workspace/paper_results', exist_ok=True)
    plt.savefig('/workspace/paper_results/birads_findings_auc_performance.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: /workspace/paper_results/birads_findings_auc_performance.png (DPI: 300)")
    plt.close()


# =============================================================================
# Print BI-RADS Finding Analysis Table
# =============================================================================

def print_birads_findings_table():
    """
    Print detailed BI-RADS findings classification metrics from analysis table
    """
    print("\n" + "="*130)
    print("BI-RADS 5-CLASS CLASSIFICATION - FINDING TYPE ANALYSIS (HARD CASES)")
    print("="*130 + "\n")
    
    print(f"{'Finding Type':<20} {'Model':<15} {'N':<5} {'Correct':<10} {'Incorrect':<12} {'Accuracy':<12} {'F1 Score':<12} {'Recall':<12} {'AUC':<8}")
    print("-" * 130)
    
    for finding in ['Mass', 'Calcification', 'Structural', 'Lymph Node']:
        effnet = findings_birads_data['EffNet-B3'][finding]
        convnext = findings_birads_data['ConvNeXt-B'][finding]
        
        # EfficientNet-B3
        print(f"{finding:<20} {'EffNet-B3':<15} {effnet['N']:<5} {effnet['Correct']:<10} {effnet['Incorrect']:<12} {effnet['Accuracy']:<12.4f} {effnet['F1']:<12.4f} {effnet['Recall']:<12.4f} {effnet['AUC']:<8.4f}")
        
        # ConvNeXt-Base
        print(f"{'':<20} {'ConvNeXt-B':<15} {convnext['N']:<5} {convnext['Correct']:<10} {convnext['Incorrect']:<12} {convnext['Accuracy']:<12.4f} {convnext['F1']:<12.4f} {convnext['Recall']:<12.4f} {convnext['AUC']:<8.4f}")
        print("-" * 130)
    
    print("\nMetrics Definition:")
    print("  • Accuracy:   Percentage of correct predictions (Correct / Total)")
    print("  • F1 Score:   Harmonic mean of Precision and Recall (Macro-F1)")
    print("  • Recall:     True Positive Rate (Correct Positives / All Positives)")
    print("  • AUC:        Area Under the Receiver Operating Characteristic Curve (0.5-1.0, higher is better)")
    print("="*130 + "\n")


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("GENERATING STRATIFIED PERFORMANCE VISUALIZATIONS")
    print("="*70 + "\n")
    
    print("Binary Classification Visualizations:")
    plot_binary_device()
    plot_binary_density()
    plot_binary_findings()
    
    print("\nBI-RADS 5-Class Classification Visualizations:")
    plot_birads_device()
    plot_birads_density()
    plot_birads_findings_accuracy()
    plot_birads_findings_f1()
    plot_birads_findings_auc()
    
    print("\n" + "="*70)
    print("✓ All visualizations generated successfully!")
    print("="*70 + "\n")
    print("Output directory: /workspace/paper_results/")
    print("\nGenerated Chart Files:")
    print("  BINARY CLASSIFICATION:")
    print("    1. binary_device_performance.png")
    print("    2. binary_density_performance.png")
    print("    3. binary_findings_f1_performance.png")
    print("  BI-RADS 5-CLASS CLASSIFICATION:")
    print("    4. birads_device_performance.png")
    print("    5. birads_density_performance.png")
    print("    6. birads_findings_accuracy_performance.png")
    print("    7. birads_findings_f1_performance.png")
    print("    8. birads_findings_auc_performance.png")
    
    # Print enhanced BI-RADS findings table
    print_birads_findings_table()


GENERATING STRATIFIED PERFORMANCE VISUALIZATIONS

Binary Classification Visualizations:
✓ Saved: /workspace/paper_results/binary_device_performance.png (DPI: 300)
✓ Saved: /workspace/paper_results/binary_density_performance.png (DPI: 300)
✓ Saved: /workspace/paper_results/binary_findings_f1_performance.png (DPI: 300)

BI-RADS 5-Class Classification Visualizations:
✓ Saved: /workspace/paper_results/birads_device_performance.png (DPI: 300)
✓ Saved: /workspace/paper_results/birads_density_performance.png (DPI: 300)
✓ Saved: /workspace/paper_results/birads_findings_accuracy_performance.png (DPI: 300)
✓ Saved: /workspace/paper_results/birads_findings_f1_performance.png (DPI: 300)
✓ Saved: /workspace/paper_results/birads_findings_auc_performance.png (DPI: 300)

✓ All visualizations generated successfully!

Output directory: /workspace/paper_results/

Generated Chart Files:
  BINARY CLASSIFICATION:
    1. binary_device_performance.png
    2. binary_density_performance.png
    3. binary_findi

In [ ]:
mkdir /workspace/gradcam_outputs